## Retrieving data from FDB

In [1]:
import earthkit.data

FDB (Fields DataBase) is a domain-specific object store developed at ECMWF for storing, indexing and retrieving GRIB data. For more information on FBD please consult the following pages:

- [FDB](https://fields-database.readthedocs.io/en/latest/)
- [pyfdb](https://pyfdb.readthedocs.io/en/latest/)

This example requires FDB access and the <b>FDB_HOME</b> environment variable has to be set correctly. 

The following request was  written to retrieve data from the operational FDB at ECMWF.  Please note that the **date** must be adjusted since FDB at ECMWF only stores the most recent dates.

In [2]:
request = {
    "class": "od",
    "expver": "0001",
    "stream": "oper",
    "date": "20260423",
    "time": [0, 12],
    "domain": "g",
    "type": "an",
    "levtype": "sfc",
    "step": 0,
    "param": [151, 167, 168],
}

### Reading as a stream

#### Iteration with one field at a time in memory

In [3]:
ds = earthkit.data.from_source("fdb", request=request).to_fieldlist()
for f in ds:
    print(f)

Field(msl, 2026-04-23 00:00:00, 2026-04-23 00:00:00, 0:00:00, 0, surface, 0, reduced_gg)
Field(2t, 2026-04-23 00:00:00, 2026-04-23 00:00:00, 0:00:00, 0, surface, 0, reduced_gg)
Field(2d, 2026-04-23 00:00:00, 2026-04-23 00:00:00, 0:00:00, 0, surface, 0, reduced_gg)
Field(msl, 2026-04-23 12:00:00, 2026-04-23 12:00:00, 0:00:00, 0, surface, 0, reduced_gg)
Field(2t, 2026-04-23 12:00:00, 2026-04-23 12:00:00, 0:00:00, 0, surface, 0, reduced_gg)
Field(2d, 2026-04-23 12:00:00, 2026-04-23 12:00:00, 0:00:00, 0, surface, 0, reduced_gg)


Once the iteration is completed, there is nothing left in *ds*.

In [4]:
sum([1 for _ in ds])

0

#### Iteration with group_by

In [5]:
ds = earthkit.data.from_source("fdb", request=request).to_fieldlist()
for f in ds.group_by("metadata.time"):
    print(f"len={len(f)} {f.metadata(('param', 'level'))}")

len=3 [('msl', 0), ('2t', 0), ('2d', 0)]
len=3 [('msl', 0), ('2t', 0), ('2d', 0)]


#### Iteration with batched

In [6]:
ds = earthkit.data.from_source("fdb", request=request).to_fieldlist()
for f in ds.batched(2):
    print(f"len={len(f)} {f.metadata(('param', 'level'))}")

len=2 [('msl', 0), ('2t', 0)]
len=2 [('2d', 0), ('msl', 0)]
len=2 [('2t', 0), ('2d', 0)]


#### Storing all the fields in memory

In [7]:
ds = earthkit.data.from_source("fdb", request=request).to_fieldlist(read_all=True)

In [8]:
len(ds)

6

In [9]:
ds.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,msl,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
1,2t,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
2,2d,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
3,msl,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg
4,2t,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg
5,2d,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg


In [10]:
ds.sel({"parameter.variable": "2t"}).ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,2t,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
1,2t,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg


In [11]:
ds.to_xarray()

<xarray.Dataset> Size: 422MB
Dimensions:                  (forecast_reference_time: 2, values: 6599680)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 16B 202...
    latitude                 (values) float64 53MB ...
    longitude                (values) float64 53MB ...
Dimensions without coordinates: values
Data variables:
    2d                       (forecast_reference_time, values) float64 106MB ...
    2t                       (forecast_reference_time, values) float64 106MB ...
    msl                      (forecast_reference_time, values) float64 106MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### Reading into a file

In [12]:
ds = earthkit.data.from_source("fdb", request=request, stream=False).to_fieldlist()

In [13]:
ds.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,msl,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
1,2t,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
2,2d,2026-04-23 00:00:00,2026-04-23 00:00:00,0 days,0,surface,0,reduced_gg
3,msl,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg
4,2t,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg
5,2d,2026-04-23 12:00:00,2026-04-23 12:00:00,0 days,0,surface,0,reduced_gg
